# Scienceverse Platform — Python API Client Demo

This notebook demonstrates how to use the Scienceverse platform API from Python.
This is a separate Scienceverse deployment with its own `/jobs` API and
credentials. It is not required to run bibr locally or host `bibr serve`.

For direct bibr API access (synchronous, no queue), see [the bibr API notebook](../../notebooks/python_api_demo.ipynb).

**Prerequisites:**
- Platform API running (or deployed at `platform.metacheck.app`)
- A platform API key (`sv_...`) — create one via SSO at the platform dashboard
- `httpx` and `pandas` installed: `uv pip install httpx pandas`

## Configuration

In [ ]:
import os
import time

import httpx

API_URL = os.getenv("PLATFORM_API_URL", "https://platform.metacheck.app")
API_KEY = os.getenv("PLATFORM_API_KEY", "").strip()
FILE_PATH = os.getenv("PAPER_PATH", "paper.pdf")

POLL_INTERVAL = 2  # seconds between polls
TIMEOUT = 600  # max seconds to wait

if not API_KEY:
    raise ValueError("Set PLATFORM_API_KEY before running this platform client")
assert os.path.exists(FILE_PATH), f"File not found: {FILE_PATH}"

headers = {"Authorization": f"Bearer {API_KEY}"}
client = httpx.Client(base_url=API_URL, headers=headers, timeout=60)

## 1. Health check

Verify the platform API is running and can reach its backends.

In [ ]:
health = client.get("/health")
print(f"Health: {health.json()}")

ready = client.get("/ready")
print(f"Ready:  {ready.json()}")

## 2. Submit a job

Upload a file to the platform queue. The response returns immediately with a
`job_id` — the file is processed asynchronously by Arq workers.

In [ ]:
filename = os.path.basename(FILE_PATH)
print(f"Submitting {filename} ...")

with open(FILE_PATH, "rb") as f:
    resp = client.post("/jobs", files={"file": (filename, f)})

assert resp.status_code == 200, f"Submit failed ({resp.status_code}): {resp.text}"

job = resp.json()
job_id = job["job_id"]
print(f"Job ID: {job_id}")
print(f"Status: {job['status']}")

## 3. Poll for completion

The platform tracks job progress through stages: `reading_upload` →
`processing` → `complete` (or `error:*` on failure).

In [ ]:
elapsed = 0.0
last_stage = None

while elapsed < TIMEOUT:
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

    status_resp = client.get(f"/jobs/{job_id}")
    status = status_resp.json()
    stage = status.get("stage", "")

    if stage != last_stage:
        suffix = f" ({stage})" if stage else ""
        print(f"  [{elapsed:5.1f}s] {status['status']}{suffix}")
        last_stage = stage

    if status["status"] == "complete":
        print(f"\nCompleted in {elapsed:.0f}s")
        break
    elif status["status"] == "failed":
        raise RuntimeError(f"Job failed: {status.get('stage', 'unknown')}")
else:
    raise TimeoutError(f"Timed out after {TIMEOUT}s")

## 4. Download the result

Results are returned as structured JSON.

In [ ]:
result_resp = client.get(
    f"/jobs/{job_id}/result",
    params={"format": "json"},
    timeout=120,
)

assert result_resp.status_code == 200, f"Download failed: {result_resp.status_code}"
data = result_resp.json()
print(f"Downloaded JSON ({len(result_resp.content):,} bytes)")

## 5. Inspect the extracted data

In [ ]:
import pandas as pd

paper_id = data["paper_id"]
metadata = data.get("metadata") or data.get("info") or {}
authors = data["author"]
text = data["text"]
sections = data["section"]
bib = data["bib"]
urls = data["url"]
xrefs = data["xref"]
figs = data["figure"]
tbls = data["table"]

print(f"Paper ID:     {paper_id}")
print(
    f"bibr version: {((data.get('extraction') or {}).get('producer') or {}).get('version') or (data.get('extraction') or {}).get('bibr_version') or metadata.get('bibr_version', 'unknown')}"
)
print(f"Sentences:    {len(text)}")
print(f"Sections:     {len(sections)}")
print(f"Authors:      {len(authors)}")
print(f"References:   {len(bib)}")
print(f"URLs:         {len(urls)}")
print(f"Xrefs:        {len(xrefs)}")
print(f"Figures:      {len(figs)}")
print(f"Tables:       {len(tbls)}")

### Paper metadata

In [ ]:
pd.DataFrame([metadata])

### Authors

In [ ]:
pd.DataFrame(authors)

### Sections

In [ ]:
pd.DataFrame(sections)

### Text (first 10 sentences)

In [ ]:
pd.DataFrame(text[:10])

### References (first 10)

In [ ]:
pd.DataFrame(bib[:10])

### Cross-references

In [ ]:
pd.DataFrame(xrefs[:10])

### Tables

In [ ]:
for t in tbls:
    print(f"--- Table {t['table_id']} (section {t['section_id']}) ---")
    contents = t.get("contents")
    if contents:
        df = pd.DataFrame(contents[1:], columns=contents[0])
        display(df.head())
    else:
        print("  (no data)")

## 6. List your recent jobs

In [ ]:
import pandas as pd

jobs_resp = client.get("/jobs", params={"limit": 10})
jobs = jobs_resp.json()

pd.DataFrame(jobs)[["job_id", "filename", "status", "created_at"]]

## 7. Save the JSON to disk

In [ ]:
import json

output_path = os.path.splitext(os.path.basename(FILE_PATH))[0] + ".json"

with open(output_path, "w") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved to {output_path}")

## 8. Summary

In [ ]:
print(f"File:           {os.path.basename(FILE_PATH)}")
print(f"Job ID:         {job_id}")
print(f"Processing:     {elapsed:.0f}s")
print(f"Title:          {metadata.get('title', 'N/A')}")
print(f"DOI:            {metadata.get('doi', 'N/A')}")
print(f"Sentences:      {len(text)}")
print(f"Sections:       {len(sections)}")
print(f"Authors:        {len(authors)}")
print(f"References:     {len(bib)}")
print(f"Links:          {len(urls)}")
print(f"Cross-refs:     {len(xrefs)}")
print(f"Figures:        {len(figs)}")
print(f"Tables:         {len(tbls)}")